In [31]:
import pandas as pd

# Read CSV File
path = "Firmoo-ES-Campaigns-Nov-12-2025-Nov-18-2025.csv"
df = pd.read_csv(path)

df = df.rename(columns={
    'Purchase ROAS (return on ad spend)': 'roas',
    'Amount spent (USD)': 'spend_usd',
    'Purchases': 'purchases'
})

df.head()


,Reporting starts,Reporting ends,Campaign name,Campaign delivery,Ad set budget,Ad set budget type,Attribution setting,Results,Result indicator,Reach,Impressions,Frequency,"CPM (cost per 1,000 impressions) (USD)",Cost per results,purchases,roas,spend_usd,Ends
0,2025-11-12,2025-11-18,ES-转化-品宣-博主-ASC,active,Using ad set budget,0,7-day click or 1-day view,93.0,actions:offsite_conversion.fb_pixel_purchase,112755,229208,2.032797,6.188702,15.252688,93.0,4.194974,1418.50,Ongoing
1,2025-11-12,2025-11-18,ES-转化-产品-目录,active,Using ad set budget,0,7-day click or 1-day view,83.0,actions:offsite_conversion.fb_pixel_purchase,102440,264873,2.585640,3.824625,12.205301,83.0,4.558162,1013.04,Ongoing
2,2025-11-12,2025-11-18,ES-转化-折扣季产品-目录-ASC,active,Using ad set budget,0,7-day click or 1-day view,59.0,actions:offsite_conversion.fb_pixel_purchase,135054,330299,2.445681,3.575427,20.016271,59.0,2.547267,1180.96,Ongoing
3,2025-11-12,2025-11-18,ES-转化-商品系列-欧珀,active,Using ad set budget,0,"7-day click, 1-day view, or 1-day engaged-view",68.0,actions:offsite_conversion.fb_pixel_purchase,101889,186100,1.826497,3.782268,10.351176,68.0,5.834745,703.88,Ongoing
4,2025-11-12,2025-11-18,ES-转化-商品系列-爆款,active,Using ad set budget,0,"7-day click, 1-day view, or 1-day engaged-view",146.0,actions:offsite_conversion.fb_pixel_purchase,267249,716882,2.682450,2.958911,14.528699,146.0,4.176745,2121.19,Ongoing


In [32]:
# 阈值可根据业务自由调整
good_roas = 3.0    
ok_roas   = 2.0    
min_pur   = 50     
low_spend = 300    

def classify_campaign(row):
    if row['Campaign delivery'] != 'active':
        return '非投放中'

    spend = row['spend_usd']
    pur   = row['purchases']
    roas  = row['roas']

    if spend < low_spend and pur < 30:
        return '数据不足，继续测试'

    if pd.isna(roas):
        return '无 ROAS 数据，需排查'

    if roas >= good_roas and pur >= min_pur:
        return '表现优秀，建议加预算'

    if roas >= ok_roas and pur >= 20:
        return '表现一般，可优化后继续投放'

    if roas < 1.5 and pur < 20:
        return '表现较差，建议暂停/重做'

    return '表现一般，可优化后继续投放'


In [33]:
df['decision'] = df.apply(classify_campaign, axis=1)

In [34]:
# 只分析 active 部分
active_df = df[df['Campaign delivery'] == 'active'].copy()

good_campaign = active_df[active_df['decision'] == '表现优秀，建议加预算']
ok_campaign   = active_df[active_df['decision'] == '表现一般，可优化后继续投放']
test_campaign = active_df[active_df['decision'] == '数据不足，继续测试']
bad_campaign  = active_df[active_df['decision'] == '表现较差，建议暂停/重做']


In [35]:
priority_order = [
    '表现优秀，建议加预算',
    '表现一般，可优化后继续投放',
    '数据不足，继续测试',
    '表现较差，建议暂停/重做'
]
priority_map = {label: i for i, label in enumerate(priority_order, start=1)}

In [36]:
all_campaign_view = pd.concat([
    good_campaign.assign(建议='表现优秀，建议加预算'),
    ok_campaign.assign(建议='表现一般，可优化后继续投放'),
    test_campaign.assign(建议='数据不足，继续测试'),
    bad_campaign.assign(建议='表现较差，建议暂停/重做')
])

# 排序用
all_campaign_view['priority'] = all_campaign_view['建议'].map(priority_map)

In [37]:
cols_to_show = [
    'Campaign name',
    'Campaign delivery',
    'spend_usd',
    'purchases',
    'roas',
    '建议'
]

result_weekly = (
    all_campaign_view
    .sort_values(['priority', 'roas'], ascending=[True, False])
    [cols_to_show]
    .reset_index(drop=True)
)

result_weekly

,Campaign name,Campaign delivery,spend_usd,purchases,roas,建议
0,ES-转化-商品系列-欧珀,active,703.88,68.0,5.834745,表现优秀，建议加预算
1,ES-转化-品宣-博主-ASC4,active,843.11,67.0,5.000676,表现优秀，建议加预算
2,ES-转化-产品-目录,active,1013.04,83.0,4.558162,表现优秀，建议加预算
3,ES-转化-品宣-博主-ASC,active,1418.50,93.0,4.194974,表现优秀，建议加预算
4,ES-转化-商品系列-爆款,active,2121.19,146.0,4.176745,表现优秀，建议加预算
5,ES-转化-商品系列-混合,active,847.55,52.0,3.500454,表现优秀，建议加预算
6,ES-转化-商品系列-套镜-Catalog,active,349.57,35.0,6.597048,表现一般，可优化后继续投放
7,ES-转化-商品系列-套镜,active,491.77,38.0,5.529679,表现一般，可优化后继续投放
8,ES-转化-促销季,active,348.28,32.0,5.132480,表现一般，可优化后继续投放
9,ES-转化-商品系列-太阳镜-Catalog,active,348.05,28.0,4.882143,表现一般，可优化后继续投放
